# PYTHON LEARNING NOTES
These are my notes for my own reference as I am brushing up on older skills and learning new skills in Python. As a fan of both languages, I may eventually turn some of these notes into a "Python for Stata Users" resource.

## SPECIFIC SKILLS

### Aggregating Data

Use df = df.agg() to aggregate data, much more flexible than sum. Example for syntax, where df_grouped is df where df.groupby() has been applied to create aggregation groups.

```python
# Step 1: Create groups within df to aggregate over
df_grouped = df.groupbby(['var1'], ['var2'])

# Step 2: Aggregate specific variables in specific ways!
df_agg = df_grouped.agg({
    'applications': 'sum',
    'population': 'mean',
    'unemployment_rate': 'mean'
})
```

**key .agg function options**
|Function | Meaning|
|:--- | :--- |
| 'sum' | add values |
| 'mean' | average |
| 'median' | median |
| 'count' | count rows |
| 'nunique' | count unique values |
| 'first' | take first row value |
| 'last' | take last row value |
| 'max' | maximum |
| 'min' | minimum |

### Panel Data Exploration / Pre-cleaning

**Checking for Patterns in Missingness**

This command structure lets you check a specific variable over var groups (year, entity) and apply a specific function to each group separately, one-by-one.

```python
df.groupby('grouping_variable')['variable of interest'].apply('exploratory function here')

# To check missing values for example:
df.groupby('year')['variable'].apply(lambda x: x.isna().sum())
```
This last bit basically does:
- (grouped subsets of the data are the things here).do_this_function_to_each_thing_separately(function that takes x: (sum of missing value count for each x))
- This uses a LAMBDA FUNCTION (see in general concepts below)


### Complicated Missingness Heatmaps

#### *Adding additional axis information!*
In this specific case, **per_idx** is what the missing_matrix must use (monthly data), but I want the resulting figure to show **year** markers on the per_idx axis. This is how we do it! 

**Making Year Markers in Y Axis**

1. Make index_years, a dataframe of the two relevant indices, per_idx and year here. 

    ```python
    index_years = new_df[['per_idx', 'year']].drop_duplicates().sort_values('per_idx')
    ```
2. Make year_positions, a SERIES that creates a spot in the index in the middle of each year category where the year marker can sit in the figure
    ```python
    year_positions = index_years.groupby('year').apply(lambda x : x.index.mean())
    ```
    - This creates index locations for middle-of-year markers. **next goal: adjust to cutoffs instead!**

    - *More detailed summary:* groups per_idx into year groups, takes the set of indices for each group and gets their mean, then returns a SERIES where each year has a value which is the average of the per_idx indices in the given year group.

    - This is a SERIES automatically because it has a grouping variable, year, which becomes the index of the series (because that is how .groupby works!), and a value associated with each grouping variable/index. If there were MORE values then I think it would be a dataframe.

3. Use this in the graph code:
    ```python
    plt.yticks(
    ticks=year_positions.values, # Pulls values of series (the averaged indices) as the tick mark locations
    labels=year_positions.index # Pulls index of series (year!) as the tick mark labels
    )
    ```
**Making Year CUTOFFS in Y-Axis**


## GENERAL CONCEPTS

.thing is a METHOD, which is a specific type of FUNCTION

They are not commands like in Stata because Python is OBJECT-ORIENTED

### Data Types in Pandas / Numpy
| Kind of Data | Recommended Pandas DType | Python Equivalen | When to Use |
|--- | --- |  --- |  --- |
| Integers | int64 or Int64 (nullable) | int | Whole numbers (IDs, counts).| 
| Floats | float64 or Float64 (nullable) | float | Numbers with decimals (prices, weights). |
| Booleans | bool or boolean (nullable) | bool | True/False flags. |
| Strings | string or object | str | Names, addresses, or long text. |
| Dates/Times | datetime64[ns] | datetime.datetime | Specific points in time. |
| Durations | timedelta64[ns] | datetime.timedelta | Time differences (e.g., 5 days). |
| Categorical | category | N/A | Text with repetitive values (e.g., "High", "Low"). |

### File Paths & Working Directories

Terminal location = working directory when running most code files.
VSCode usually opens full project folder and terminal is based there, which is why .py filepaths don't need to go out of the code folder to access data folder files.

HOWEVER! Jupyter Notebooks are different and they run code from the location of the notebook file. SO filepaths need to exit the notebook folder before accessing any other files.
Do this with 
../filepath 

### Matplotlib

Unlike Stata, making graphs in python goes: 1) create figure object 2) modify figure 3) show figure

to show figures when using VSCode:
```python
    import matplotlib.pyplot as plt

    figure = plt.figure()  # creates an empty figure object
    # figure information here 
    # figure modifications here
    plt.show() # shows the figure!
```

Matplotlib documentation/guide: [link](https://matplotlib.org/stable/users/explain/quick_start.html)

#### Key Components/Terms
* Figure - a whole graph/figure object that contains all of the following components/information
* Artists - every visible component on the figure is an Artist. Most are tied to an Axes
* AXES - a region for plotting data, and 2-3 Axis objects that create the tick marks
* AXIS - 


#### Popular Color Maps (cmap = )
| cmap         | vibe                    |
| ------------ | ----------------------- |
| `'Blues'`    | clean/professional      |
| `'Greens'`   | softer                  |
| `'Reds'`     | intense                 |
| `'Greys'`    | minimalist              |
| `'cividis'`  | accessibility-friendly  |
| `'magma'`    | dark/dramatic           |
| `'coolwarm'` | diverging               |
| `'BuGn'`     | blue-green              |
| `'YlGnBu'`   | nice academic-map vibes |
| `'binary'`   | black/white             |



### Functions & Lambda Functions

**Lambda functions** are basically quick, tiny functions that just do one thing without the full function drama.

Syntax: lambda *arguments: expression*
* lambda = the keyword that starts the definition of this type of function
* arguments = the inputs to the function (x, y, whatever)(can be one, can be multiple)
    * how to define multiple?
* expression = whatever expression you want the function to do




### Data Objects in Python

**Lists vs. Tuples**

Tuple = 'thing1', 'thing2', 'etc'

List = ['thing1', 'thing2', 'etc']

You can loop over items in a tuple.

You can put a list into a command and have it use each item of the list at once.
* ex: quarters must be a list here, not a tuple, to add all the columns referenced by the list together at once

    ```python 
    formations['state_formations'] = formations[quarters].sum(axis=1)
    ```
    

### Modifying Strings in Python (using str class)
Useful page: https://www.w3schools.com/python/python_strings_modify.asp


1. Create a list of text objects by splitting a string on a certain character:
    ```python 
    text = "apple-banana-watermelon
    fruits = text.split("-") # splits over the specified character
    # returns ['apple', 'banana', 'watermelon']
    ```

2. Slicing!!!
    - use [:#] format
    - number is the first character number (starting from zero) that you want to be excluded
    - colon before or after number to get text before or after that character 
    ```python
    text = "apple and banana"

    apple = text[:6]
    # returns 'apple'

    banana = text[10:]
    # returns 'banana'
    
    word = text[6:10] # grabs characters BETWEEN 6 and 10
    # returns 'and'
    ```